# Week 2 – Builder (Cemre Gönenç): EasyOCR vs PaddleOCR

**Goal:** Run the 50 plate crops through both OCR tools, count correct readings, and pick the better tool.

**First:** Runtime → Change runtime type → **T4 GPU** → Save.
Run the cells **one by one** (Shift + Enter).

## 1) Install the OCR tools
Takes 2–3 minutes. If the next cells give an import error, use **Runtime → Restart session** and continue from step 2 (no need to install again).

In [ ]:
!pip install -q easyocr paddlepaddle paddleocr

## 2) Get the 50 plates from GitHub + mount Drive

In [ ]:
import os
if not os.path.exists('anpr-project-en'):
    !git clone -q https://github.com/cemregonenc/anpr-project-en.git
PLATES = 'anpr-project-en/data/plates'

from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/ANPR/results'
os.makedirs(RESULTS, exist_ok=True)

import pandas as pd
labels = pd.read_csv(f'{PLATES}/labels.csv')
print(len(labels), 'plates')
labels.head()

## 3) Text cleaning for a fair comparison
Both tools return raw text like `"34 PBL 657"` or `"TR 34PBL-657"`. Before comparing with `labels.csv` we:
- make it uppercase,
- keep only letters and digits,
- remove the `TR` from the blue country band if it was read.

This is **not** the Week 3 cleanup (no O→0 fixes yet). We only remove spaces and symbols.

In [ ]:
import re

def normalize(text):
    t = re.sub(r'[^A-Z0-9]', '', str(text).upper())
    # Turkish plates start with 2 digits; drop a leading 'TR' read from the blue band
    if t.startswith('TR') and t[2:4].isdigit():
        t = t[2:]
    return t

print(normalize('TR 34 PBL-657'), normalize('06 dJp 36'))

## 4) EasyOCR

In [ ]:
import easyocr, time

reader = easyocr.Reader(['en'], gpu=True)

easy_out, start = [], time.time()
for f in labels['filename']:
    texts = reader.readtext(f'{PLATES}/{f}', detail=0)   # list of text pieces
    easy_out.append(' '.join(texts))
easy_time = time.time() - start
labels['easyocr_raw'] = easy_out
print(f'EasyOCR: {easy_time:.1f} s for {len(labels)} plates')
labels[['filename', 'plate_text', 'easyocr_raw']].head(10)

## 5) PaddleOCR

In [ ]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='en',
                use_doc_orientation_classify=False,
                use_doc_unwarping=False,
                use_textline_orientation=False)

def paddle_read(path):
    res = ocr.predict(path)
    texts = []
    for r in res:
        texts += list(r['rec_texts'])
    return ' '.join(texts)

paddle_out, start = [], time.time()
for f in labels['filename']:
    paddle_out.append(paddle_read(f'{PLATES}/{f}'))
paddle_time = time.time() - start
labels['paddle_raw'] = paddle_out
print(f'PaddleOCR: {paddle_time:.1f} s for {len(labels)} plates')
labels[['filename', 'plate_text', 'paddle_raw']].head(10)

## 6) Score both tools
- **Exact match:** the whole plate is correct (this is the number that matters).
- **Character similarity:** how close the reading is (0–100%). Shows *near misses* such as `34PBL657` vs `34PBL6S7`.

In [ ]:
from difflib import SequenceMatcher

def sim(a, b):
    return SequenceMatcher(None, a, b).ratio()

for tool in ['easyocr', 'paddle']:
    labels[f'{tool}_clean'] = labels[f'{tool}_raw'].apply(normalize)
    labels[f'{tool}_ok'] = labels[f'{tool}_clean'] == labels['plate_text']
    labels[f'{tool}_sim'] = [sim(a, b) for a, b in zip(labels[f'{tool}_clean'], labels['plate_text'])]

summary = pd.DataFrame({
    'Tool': ['EasyOCR', 'PaddleOCR'],
    'Correct (of 50)': [labels['easyocr_ok'].sum(), labels['paddle_ok'].sum()],
    'Accuracy %': [100 * labels['easyocr_ok'].mean(), 100 * labels['paddle_ok'].mean()],
    'Avg char similarity %': [100 * labels['easyocr_sim'].mean(), 100 * labels['paddle_sim'].mean()],
    'Time (s)': [easy_time, paddle_time],
}).round(1)
summary

### Accuracy by condition (clear, blurry, small, angled …)

In [ ]:
by_cond = labels.groupby('condition').agg(
    plates=('filename', 'count'),
    easyocr_correct=('easyocr_ok', 'sum'),
    paddle_correct=('paddle_ok', 'sum'))
by_cond.sort_values('plates', ascending=False)

## 7) Look at the mistakes
These examples go into the report and tell us what to fix in Week 3.

In [ ]:
import matplotlib.pyplot as plt, cv2

wrong = labels[~labels['easyocr_ok'] | ~labels['paddle_ok']]
print(len(wrong), 'plates where at least one tool was wrong')

n = len(wrong)
if n:
    cols = 3; rows = (n + cols - 1) // cols
    plt.figure(figsize=(15, 2.6 * rows))
    for i, (_, r) in enumerate(wrong.iterrows()):
        img = cv2.cvtColor(cv2.imread(f"{PLATES}/{r['filename']}"), cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i + 1); plt.imshow(img); plt.axis('off')
        plt.title(f"truth {r['plate_text']}\nEasy {r['easyocr_clean'] or '-'} | Paddle {r['paddle_clean'] or '-'}", fontsize=9)
    plt.tight_layout(); plt.show()

### Most common character mix-ups
(Only for readings with the right length, so characters line up.) This list becomes the Week 3 fix rules, e.g. O→0, S→5.

In [ ]:
from collections import Counter

def confusions(col):
    c = Counter()
    for t, p in zip(labels['plate_text'], labels[col]):
        if len(t) == len(p):
            c.update(f'{a}→{b}' for a, b in zip(t, p) if a != b)
    return c.most_common(10)

print('EasyOCR  :', confusions('easyocr_clean'))
print('PaddleOCR:', confusions('paddle_clean'))

## 8) Save results to Drive

In [ ]:
out = f'{RESULTS}/week2_ocr_compare.csv'
labels.to_csv(out, index=False)
summary.to_csv(f'{RESULTS}/week2_ocr_summary.csv', index=False)
print('Saved:', out)

best = summary.sort_values(['Correct (of 50)', 'Avg char similarity %'], ascending=False).iloc[0]
print(f"\nWinner: {best['Tool']} – {best['Correct (of 50)']}/50 correct ({best['Accuracy %']}%)")

## 9) Conclusion (fill in after running)

| Tool | Correct (of 50) | Accuracy | Avg similarity | Time |
|---|---|---|---|---|
| EasyOCR | | | | |
| PaddleOCR | | | | |

**Chosen tool:** …
**Why:** …
**Hardest conditions:** …
**Most common mistakes (Week 3 fixes):** …